# Experiment Tracking & Versioning

**Course:** [ML in Practice](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/07-experiment-tracking)

A run is a black box that consumed code, data, and a stack of GPUs. The point of this notebook is to make that box transparent. We build a tiny in-memory experiment tracker, run a hyperparameter sweep through it, version the training data with content-addressed hashing, and replay the best run from scratch — proving that *with* the right metadata, a run is reproducible.

1. **A minimal `ExperimentTracker`.** `start_run(params)`, `log_metric(name, value, step)`, `log_artifact(name, payload)`, `end_run()`. Persists to a JSON file so a notebook restart is fine.
2. **A 30-run hyperparameter sweep.** Ten learning rates times three seeds on a synthetic regression problem. Log loss curves and final RMSE.
3. **Plot all 30 loss curves.** Coloured by LR. Mark the best run.
4. **Content-addressed data versioning.** A `data_hash(array)` helper that hashes the bytes + shape + dtype. Show that two splits of the same dataset have different hashes; the same split is bit-identical.
5. **Reproducibility demo.** Replay the best run from its logged params + data hash and confirm the final weights and metrics match the first run to numerical precision.
6. **Discussion.** What this toy tracker does *not* do — and why MLflow / Weights & Biases / DVC exist.

Self-contained: NumPy + matplotlib + `hashlib` from the standard library. No sklearn, no MLflow, no W&B, no network, no API keys.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import hashlib
import json
import time
import uuid
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2a2d3a',
    'axes.labelcolor':  '#e2e8f0',
    'text.color':       '#e2e8f0',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'grid.color':       '#2a2d3a',
    'grid.alpha':       0.5,
})

BRAND  = '#6366f1'
TEAL   = '#2dd4bf'
ROSE   = '#fb7185'
ORANGE = '#f97316'
YELLOW = '#facc15'
MUTED  = '#475569'

## 1. A minimal `ExperimentTracker`

The schema is intentionally tiny:

- `runs` — a dict keyed by `run_id`. Each run holds its parameters, metrics (a `name → list of (step, value)` map), and artifacts (a `name → small payload` map).
- `start_run(params)` returns a `run_id` (a UUID); `log_metric` and `log_artifact` write into that run; `end_run` stamps the end time.

This is the same shape MLflow uses under the hood. The persistence layer is a single JSON file — sufficient for a one-machine demo, woefully insufficient for a real team (no concurrent writers, no query layer, no artifact deduplication). We will name those limitations explicitly at the end of the notebook.

In [ ]:
class ExperimentTracker:
    '''Toy in-memory experiment tracker, persisted to one JSON file.'''

    def __init__(self, path='runs.json'):
        self.path = Path(path)
        self.runs = {}
        self._current = None

    def start_run(self, params, tags=None):
        run_id = uuid.uuid4().hex[:8]
        self.runs[run_id] = {
            'run_id': run_id,
            'started_at': time.time(),
            'ended_at': None,
            'params': dict(params),
            'tags': dict(tags or {}),
            'metrics': {},        # name -> list of (step, value)
            'summary': {},        # name -> scalar
            'artifacts': {},      # name -> payload (small numeric or string)
        }
        self._current = run_id
        return run_id

    def log_metric(self, name, value, step):
        assert self._current is not None, 'no active run'
        run = self.runs[self._current]
        run['metrics'].setdefault(name, []).append((int(step), float(value)))

    def log_summary(self, name, value):
        assert self._current is not None, 'no active run'
        self.runs[self._current]['summary'][name] = float(value)

    def log_artifact(self, name, payload):
        assert self._current is not None, 'no active run'
        # In a real tracker, payload is a file handle and the store hashes the bytes.
        # Here we just stash a small JSON-serialisable thing.
        self.runs[self._current]['artifacts'][name] = payload

    def end_run(self):
        assert self._current is not None
        self.runs[self._current]['ended_at'] = time.time()
        self._current = None
        self._persist()

    def _persist(self):
        with self.path.open('w') as f:
            json.dump(self.runs, f, indent=2, default=str)

    def best_run(self, summary_metric, mode='min'):
        scored = [(r['summary'].get(summary_metric, np.inf if mode == 'min' else -np.inf), r)
                  for r in self.runs.values() if summary_metric in r['summary']]
        if not scored:
            return None
        scored.sort(key=lambda t: t[0], reverse=(mode == 'max'))
        return scored[0][1]


tracker = ExperimentTracker(path='runs.json')
print(f'Tracker ready. Persistence path: {tracker.path.resolve()}')

## 2. A 30-run hyperparameter sweep

Synthetic regression: $y = X \beta + \varepsilon$ with $\beta = [1.5, -2.0, 0.7]$ and Gaussian noise. We sweep 10 learning rates from $10^{-3}$ to $10^{-1}$ on a log grid, three seeds each, for 30 runs total. Each run logs a loss curve and a final RMSE.

In [ ]:
# Build a deterministic synthetic regression dataset.
TRUE_BETA = np.array([1.5, -2.0, 0.7])

def make_dataset(n=400, seed=0):
    r = np.random.default_rng(seed)
    X = r.normal(size=(n, 3))
    eps = r.normal(scale=0.3, size=n)
    y = X @ TRUE_BETA + eps
    return X, y


X_all, y_all = make_dataset(n=400, seed=0)
# Use a fixed 80/20 train/test split.
N = len(y_all)
perm = np.random.default_rng(123).permutation(N)
n_tr = int(0.8 * N)
tr_idx, te_idx = perm[:n_tr], perm[n_tr:]
X_tr, y_tr = X_all[tr_idx], y_all[tr_idx]
X_te, y_te = X_all[te_idx], y_all[te_idx]
print(f'train: {X_tr.shape}, test: {X_te.shape}')

In [ ]:
def train_linreg(X, y, lr, n_iters=200, seed=0):
    '''Gradient-descent linear regression. Returns weights and loss curve.'''
    r = np.random.default_rng(seed)
    n, d = X.shape
    Xb = np.hstack([X, np.ones((n, 1))])
    w = r.normal(scale=0.05, size=d + 1)
    loss_curve = []
    for step in range(n_iters):
        yhat = Xb @ w
        resid = yhat - y
        loss = 0.5 * float(np.mean(resid ** 2))
        loss_curve.append(loss)
        grad = (Xb * resid[:, None]).mean(axis=0)
        w = w - lr * grad
    return w, loss_curve


def rmse(w, X, y):
    Xb = np.hstack([X, np.ones((len(X), 1))])
    return float(np.sqrt(np.mean((Xb @ w - y) ** 2)))


# Sweep.
LRS = np.logspace(-3, -1, 10)
SEEDS = [0, 1, 2]

run_ids = []
for lr in LRS:
    for seed in SEEDS:
        run_id = tracker.start_run(
            params={'lr': float(lr), 'seed': int(seed), 'n_iters': 200, 'd': 3},
            tags={'experiment': 'lr-sweep'},
        )
        w, loss_curve = train_linreg(X_tr, y_tr, lr=lr, n_iters=200, seed=seed)
        for step, l in enumerate(loss_curve):
            tracker.log_metric('train_loss', l, step)
        final_rmse = rmse(w, X_te, y_te)
        tracker.log_summary('test_rmse', final_rmse)
        tracker.log_summary('final_train_loss', loss_curve[-1])
        tracker.log_artifact('weights', w.tolist())
        tracker.end_run()
        run_ids.append(run_id)

print(f'Logged {len(run_ids)} runs to {tracker.path}')

# Top 3 by test RMSE.
ranked = sorted(tracker.runs.values(), key=lambda r: r['summary']['test_rmse'])
print('\nTop 3 runs by test RMSE:')
for r in ranked[:3]:
    p = r['params']
    print(f"  run {r['run_id']}  lr={p['lr']:.4f}  seed={p['seed']}  test_rmse={r['summary']['test_rmse']:.4f}")

## 3. Plot all 30 loss curves

The whole point of logging *every* run, not just the winner: we get to see the shape of the failure modes. Curves are coloured by LR (log scale). The best run is highlighted in teal.

In [ ]:
best = tracker.best_run('test_rmse', mode='min')
print(f"best run: {best['run_id']}  lr={best['params']['lr']:.4f}  seed={best['params']['seed']}  test_rmse={best['summary']['test_rmse']:.4f}")

cmap = plt.cm.viridis
log_lrs = np.log10(LRS)
norm = plt.Normalize(log_lrs.min(), log_lrs.max())

fig, ax = plt.subplots(figsize=(8.5, 5))
for r in tracker.runs.values():
    steps, losses = zip(*r['metrics']['train_loss'])
    lr = r['params']['lr']
    color = cmap(norm(np.log10(lr)))
    is_best = (r['run_id'] == best['run_id'])
    ax.plot(steps, losses, color=color, alpha=0.5 if not is_best else 1.0,
            linewidth=2.5 if is_best else 1.0,
            label=f"BEST  lr={lr:.4f}, seed={r['params']['seed']}" if is_best else None)

ax.set_yscale('log')
ax.set_xlabel('step')
ax.set_ylabel('train loss (log)')
ax.set_title('30-run LR sweep — all loss curves coloured by LR; best run highlighted')
ax.grid(True)
ax.legend(frameon=False, loc='upper right')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02)
cbar.set_label('log10(lr)')
plt.tight_layout(); plt.show()

Notice the shape: low-LR runs (purple, bottom of the colorbar) descend slowly and have not converged in 200 steps; high-LR runs (yellow) oscillate or diverge; the best runs sit in a tight middle band. This is the canonical learning-rate sweep figure, and the *failed* runs at the extremes are exactly what teach the team where the band is — which is why we always log every run, not just the winner.

## 4. Content-addressed data versioning

A *path* like `s3://mybucket/training/v3/` is not a version — its bytes can be overwritten silently. A content-addressed version is the *hash of the bytes*. Two different splits of the same dataset get different hashes; the same split is bit-identical and hashes to the same value.

The reshape gotcha: `arr.tobytes()` ignores `arr.shape` and `arr.dtype`, so a reshape that does not change the underlying byte order has the same `tobytes()`. A robust data hash must include the shape and dtype too — otherwise a `(400, 3)` matrix and a `(1200,)` flattened version collide.

In [ ]:
def data_hash(arr):
    '''Content-addressed SHA-256 of a NumPy array. Includes shape and dtype.'''
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(arr).tobytes())
    h.update(str(arr.shape).encode())
    h.update(str(arr.dtype).encode())
    return h.hexdigest()


# Two different splits of the same dataset -> different hashes.
split_a_idx = np.random.default_rng(0).permutation(N)[:n_tr]
split_b_idx = np.random.default_rng(7).permutation(N)[:n_tr]

ha = data_hash(X_all[split_a_idx])
hb = data_hash(X_all[split_b_idx])

# The same split, recomputed -> same hash.
ha_again = data_hash(X_all[np.random.default_rng(0).permutation(N)[:n_tr]])

# The training split we actually used:
htrain = data_hash(X_tr)

print('hashes (first 16 chars):')
print(f'  split A         : {ha[:16]}')
print(f'  split B         : {hb[:16]}')
print(f'  split A again   : {ha_again[:16]}  (matches A: {ha == ha_again})')
print(f'  X_tr (our split): {htrain[:16]}')
print(f'  A == B          : {ha == hb}  (should be False)')

## 5. Reproducibility demo: replay the best run

We pretend we are coming back to the project six weeks later. We know nothing except what the tracker recorded for the best run:

- the **params** (lr, seed, n_iters),
- the **data hash** the run consumed.

We re-train from scratch with exactly those params on data whose hash matches, and confirm the resulting weights and metrics match the first run to numerical precision.

In [ ]:
# Step 1: record the data hash on the original best run (in real life this
# would have been logged at training time).
best['tags']['data_hash'] = htrain
best['artifacts']['data_hash'] = htrain

original_weights = np.array(best['artifacts']['weights'])
original_rmse = best['summary']['test_rmse']
original_lr = best['params']['lr']
original_seed = best['params']['seed']
original_n_iters = best['params']['n_iters']
original_data_hash = best['artifacts']['data_hash']

print('Replay using only what the tracker recorded:')
print(f'  lr        = {original_lr}')
print(f'  seed      = {original_seed}')
print(f'  n_iters   = {original_n_iters}')
print(f'  data_hash = {original_data_hash[:16]}...')
print()

# Step 2: load the dataset and confirm its hash matches.
replay_hash = data_hash(X_tr)
assert replay_hash == original_data_hash, 'data has changed since the original run!'
print(f'data hash OK: {replay_hash[:16]}... matches recorded hash')

# Step 3: re-train with the same params.
replay_w, _ = train_linreg(X_tr, y_tr, lr=original_lr, n_iters=original_n_iters, seed=original_seed)
replay_rmse = rmse(replay_w, X_te, y_te)

# Step 4: compare.
weight_diff = float(np.max(np.abs(replay_w - original_weights)))
rmse_diff = abs(replay_rmse - original_rmse)
print(f'\nmax|weight delta|  = {weight_diff:.2e}')
print(f'|rmse delta|       = {rmse_diff:.2e}')
print(f'replay reproduces original to numerical precision: {weight_diff < 1e-12 and rmse_diff < 1e-12}')

The replay is bit-identical because every source of variability was either pinned (lr, n_iters, seed) or hash-verified (data). This is exactly what a production tracker buys you. On a real ML team, the same exercise extends to:

- the **code SHA** (so the training script itself can be re-checked out),
- the **environment digest** (so the same CUDA / PyTorch / driver stack is used),
- and the **distributed config** (world size, gradient accumulation, dataloader seed *per rank*).

Forget any one of them and the replay produces a different model — not necessarily *wrong*, but not the same model.

## 6. What this toy tracker does NOT do

In ~80 lines we covered the *schema* of an experiment tracker. A production tracker buys you a long list of properties this toy does not:

- **Multi-user concurrent writes.** Real trackers use a database (Postgres, SQLite-in-server) so two simultaneous runs can both log without corrupting state.
- **A queryable UI.** "Show me every run from the last week with `lr < 0.01` sorted by test_rmse." We have a JSON file; you can `grep` it.
- **Large artifacts.** Model checkpoints are gigabytes; you do not stash them in a JSON. Real trackers stream artifacts to an object store and store only the hash + URI in the run record.
- **Artifact deduplication.** If two runs emit byte-identical weights, the artifact store should de-duplicate to one blob. Ours does not.
- **A lineage graph.** Real trackers + registries record the prediction → model → run → data graph and let you walk it. We have no graph.
- **The model registry layer.** Promotion of a run to a versioned production model is an audited, immutable event. Our toy does not draw the line.
- **Real data versioning.** Hashing the bytes of a NumPy array is the *concept*. DVC, LakeFS, Apache Iceberg, and Delta Lake are the *implementations* — they version Parquet files in object stores, replicate to disaster-recovery regions, garbage-collect old versions, and let you `git checkout` a dataset.
- **Environment fingerprinting.** A real run record pins a container digest (or at minimum a pip-tools lockfile hash). Ours pins nothing.

Use this notebook to *understand* what the tools are doing. Reach for MLflow, Weights & Biases, Sacred, ClearML, or Neptune in real work.

---
## ✏️ Your turn

### Exercise: implement `data_hash(arr)` so reshapes do not collide

A naïve content hash that only feeds `arr.tobytes()` to SHA-256 collides on reshapes: a `(400, 3)` matrix and its flattened `(1200,)` view have the same bytes, so the same hash. That collision lets a bug at preprocessing time silently produce two different "valid" datasets that the tracker thinks are the same.

Implement `data_hash_yours(arr)` that hashes the array's *bytes plus its shape plus its dtype*, so reshaping the same underlying data produces a *different* hash. (`hashlib.sha256()` from the standard library is fine.)

**Property to satisfy.** For an array `arr`:

- `data_hash_yours(arr) == data_hash_yours(arr)` — same bytes + shape + dtype → same hash.
- `data_hash_yours(arr) != data_hash_yours(arr.reshape(-1))` — same bytes, different shape → *different* hash.
- `data_hash_yours(arr) != data_hash_yours(arr.astype(np.float32))` — same logical values, different dtype → *different* hash.

In [ ]:
import hashlib  # already imported above — re-import for the exercise cell

def data_hash_yours(arr):
    '''Return a SHA-256 hex digest of `arr` that is sensitive to shape AND dtype.

    Args:
        arr : NumPy array.

    Returns:
        str hex digest. Two arrays with the same bytes but different shape
        (or different dtype) must produce different digests.
    '''
    # TODO(you):
    #   1. Make `arr` contiguous (np.ascontiguousarray) so .tobytes() is canonical.
    #   2. Create a hashlib.sha256() object.
    #   3. Update it with arr.tobytes().
    #   4. ALSO update it with str(arr.shape).encode() so reshapes do not collide.
    #   5. ALSO update it with str(arr.dtype).encode() so dtype changes do not collide.
    #   6. Return h.hexdigest().
    pass


# Smoke run.
arr_demo = np.arange(12, dtype=np.float64).reshape(3, 4)
print(f'data_hash_yours(arr_demo) = {data_hash_yours(arr_demo)}')

In [ ]:
# Test 1: deterministic — same input -> same hash.
a = np.arange(12, dtype=np.float64).reshape(3, 4)
h1 = data_hash_yours(a)
h2 = data_hash_yours(np.arange(12, dtype=np.float64).reshape(3, 4))
assert h1 is not None, 'function returned None'
assert h1 == h2, 'same input should produce same hash'

# Test 2: reshape collision — same bytes, different shape -> DIFFERENT hash.
flat = a.reshape(-1)
assert a.tobytes() == flat.tobytes(), 'bytes should match (test invariant)'
h_flat = data_hash_yours(flat)
assert h1 != h_flat, 'reshape must produce a different hash; otherwise the hash is shape-blind'

# Test 3: dtype collision — same logical values, different dtype -> DIFFERENT hash.
a32 = a.astype(np.float32)
h_32 = data_hash_yours(a32)
assert h1 != h_32, 'dtype change must produce a different hash'

# Test 4: changing one element changes the hash.
a_perturbed = a.copy(); a_perturbed[0, 0] += 1e-12
h_pert = data_hash_yours(a_perturbed)
assert h1 != h_pert, 'a one-bit change in the data must change the hash'

# Test 5: hex digest format.
assert isinstance(h1, str) and len(h1) == 64 and all(c in '0123456789abcdef' for c in h1), \
    'expected a 64-character hex string from SHA-256'

print('All data_hash tests passed.')

<details>
<summary>Show solution</summary>

```python
def data_hash_yours(arr):
    h = hashlib.sha256()
    h.update(np.ascontiguousarray(arr).tobytes())
    h.update(str(arr.shape).encode())
    h.update(str(arr.dtype).encode())
    return h.hexdigest()
```

Three things to notice:

1. **`np.ascontiguousarray` is load-bearing.** A non-contiguous view's `.tobytes()` may differ from the contiguous equivalent — copying it first makes the byte representation canonical. Without this, transposed or strided views of the same data hash differently and the version system becomes a liar.
2. **Including the shape is what fixes the reshape collision.** `arr.tobytes()` is shape-blind: a `(400, 3)` matrix and its flattened `(1200,)` view have *identical* bytes. Without `str(arr.shape)` in the hash, a preprocessing bug that flattens the matrix produces the same "version" — and the run that consumes it will look reproducible while actually training on a different data structure.
3. **Including the dtype matters for floats vs ints.** `np.float64(1.0).tobytes()` and `np.float32(1.0).tobytes()` are different bytes, so dtype changes *usually* show up — but they show up "for free" via the byte representation. Pinning dtype explicitly is belt-and-braces; it also catches the edge case where two dtypes happen to produce the same bytes for a particular value (rare but real for, e.g., zero-filled arrays of compatible widths).

In a real DVC / LakeFS / Iceberg setup the hashing is done on the file (a Parquet block, a TFRecord shard), not on a NumPy array — but the principle is the same: shape, schema, and bytes all participate. A "version" that ignores any of them is a pointer, not a version.
</details>